# BKG Studies

In [1]:
# python
import sys
import os
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
# from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
# sidm_path = str(sys.path[0]).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, cutflow, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
from tqdm.notebook import tqdm
import coffea.util
from hist import Hist
import numpy as np
# from sidm.definitions.hists import constants


In [2]:
# 06: 1 files for:
# 2Mu2E_100GeV_5p0GeV_0p4mm", "2Mu2E_200GeV_5p0GeV_0p2mm", "2Mu2E_500GeV_5p0GeV_80p0mm",     "2Mu2E_1000GeV_5p0GeV_0p04mm",
# 4Mu_100GeV_5p0GeV_0p4mm", "4Mu_200GeV_5p0GeV_0p2mm", "4Mu_500GeV_5p0GeV_80p0mm", "4Mu_1000GeV_5p0GeV_0p04mm",
# "DYJetsToMuMu_M10to50",    "DYJetsToMuMu_M50",    "TTJets",     "QCD_Pt30To50",     "QCD_Pt50To80",     "QCD_Pt80To120", "QCD_Pt470To600", "QCD_Pt1000",

In [3]:
# my settings
numfiles = 3
vr = "06"

sig_2mu = [
    "2Mu2E_100GeV_5p0GeV_0p4mm",
    # "2Mu2E_150GeV_5p0GeV_0p27mm",
    # "2Mu2E_200GeV_5p0GeV_0p2mm",
    # "2Mu2E_500GeV_5p0GeV_0p08mm",
    # "2Mu2E_500GeV_5p0GeV_80p0mm",
    # "2Mu2E_800GeV_5p0GeV_0p05mm",
    # "2Mu2E_1000GeV_5p0GeV_0p04mm",
]

sig_4mu = [
    "4Mu_100GeV_5p0GeV_0p4mm",
    # "4Mu_150GeV_5p0GeV_0p27mm",
    # "4Mu_200GeV_5p0GeV_0p2mm",
    # "4Mu_500GeV_5p0GeV_0p08mm",
    # "4Mu_500GeV_5p0GeV_80p0mm",
    # "4Mu_800GeV_5p0GeV_0p05mm",
    # "4Mu_1000GeV_5p0GeV_0p04mm",
]

bkg = [
    "DYJetsToMuMu_M10to50",
    # "DYJetsToMuMu_M50",
    "TTJets",
    # "QCD_Pt15To20",
    # "QCD_Pt20To30",
    # "QCD_Pt30To50",
    # "QCD_Pt50To80",
    # "QCD_Pt80To120",
    # "QCD_Pt120To170",
    # "QCD_Pt170To300",
    # "QCD_Pt300To470",
    # "QCD_Pt470To600",
    # "QCD_Pt600To800",
    # "QCD_Pt800To1000",
    # "QCD_Pt1000",       
]

#cuts to be applied (slections.yaml)
channels = ["baseNoLj", "bkg_study", "bkg_study_iso",]

ch1 = channels[0]
ch2 = channels[1]
ch3 = channels[2]

allsamples = sig_2mu + sig_4mu + bkg
print(allsamples)

# sig_leg = [s.split("_")[1] + "  " + s.split("_")[2] + "  " + s.split("_")[3] for s in sig_2mu]
figw, figh = 10, 10

# cols = ["b", "r", "g", "c", "k", "y"]
# labels = ["0.15", "0.17", "0.19", "0.21", "0.23", "0.25"]

['2Mu2E_100GeV_5p0GeV_0p4mm', '4Mu_100GeV_5p0GeV_0p4mm', 'DYJetsToMuMu_M10to50', 'TTJets']


In [4]:
# processor for 2mu2e

fileset_sig_2mu = utilities.make_fileset(sig_2mu,  "llpNanoAOD_v2", max_files = numfiles, location_cfg = "signal_2mu2e_v10.yaml")

runner = processor.Runner(
    executor=processor.FuturesExecutor(),
    # executor=processor.IterativeExecutor(),
    # executor=processor.DaskExecutor(client=client),
    schema=llpnanoaodschema.LLPNanoAODSchema,
    # schema=NanoAODSchema,
    # maxchunks=1,
    skipbadfiles=True,
)

#hist collection
p = sidm_processor.SidmProcessor(
    channels,
    ["BKG_study"], unweighted_hist=True,
)

out_sig = runner.run(fileset_sig_2mu, treename="Events", processor_instance=p)
out_sig = out_sig["out"]

Output()

Output()

/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(
/usr/local/lib/python3.12/site-packages/awkward/_nplikes/array_module.py:285: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


concurrent.futures.process._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/coffea/processor/executor.py", line 1493, in _work_function
    out = processor_instance.process(events)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/cms-jovyan/SIDM/sidm/tools/sidm_processor.py", line 106, in process
    ch_cuts = self.build_cuts()
              ^^^^^^^^^^^^^^^^^
  File "/home/cms-jovyan/SIDM/sidm/tools/sidm_processor.py", line 325, in build_cuts
    selection_menu = utilities.load_yaml(f"{BASE_DIR}/{self.selections_cfg}")
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/cms-jovyan/SIDM/sidm/tools/utilities.py", line 173, in load_yaml
    return yaml.safe_load(yaml_cfg)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/yaml/__init__.py", line 125, in safe_load
    return load(stream, SafeLoader)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/loca

Exception: Failed processing file: WorkItem(dataset='2Mu2E_100GeV_5p0GeV_0p4mm', filename='root://xcache//store/group/lpcmetx/SIDM/ULSignalSamples/2018_v10/BsTo2DpTo2Mu2e/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-100_MDp-5p0_ctau-0p4_v3/LLPnanoAODv2/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-100_MDp-5p0_ctau-0p4_v3_part-0.root', treename='Events', entrystart=0, entrystop=2039, fileuuid=b'@\x86h6\xbe\xae\x11\xef\x94\xc4m\x15\xe6\x9b\xbe\xef', usermeta={'skim_factor': 1.0, 'year': '2018', 'is_data': False}). The error was: ParserError('while parsing a block collection', <yaml.error.Mark object at 0x7ff0df2b4200>, "expected <block end>, but found '<scalar>'", <yaml.error.Mark object at 0x7ff0df2b4110>).

In [ ]:
# processor for 4mu signal

fileset_sig_4mu = utilities.make_fileset(sig_4mu,  "llpNanoAOD_v2", max_files = numfiles, location_cfg = "signal_4mu_v10.yaml")

runner = processor.Runner(
    executor=processor.FuturesExecutor(),
    # executor=processor.IterativeExecutor(),
    # executor=processor.DaskExecutor(client=client),
    # schema=NanoAODSchema,
    schema=llpnanoaodschema.LLPNanoAODSchema,
    # maxchunks=1,
    skipbadfiles=True,
)

#hist collection
p = sidm_processor.SidmProcessor(
    channels,
    ["BKG_study"], unweighted_hist=True,
)

out_sig4 = runner.run(fileset_sig_4mu, treename="Events", processor_instance=p)
out_sig4 = out_sig4["out"]

In [ ]:
# processor for BKG ...

fileset_bkg    = utilities.make_fileset(bkg, "llpNanoAOD_v2", max_files = numfiles, location_cfg = "backgrounds.yaml",)

runner = processor.Runner(
    executor=processor.FuturesExecutor(),
    # executor=processor.IterativeExecutor(),
    # executor=processor.DaskExecutor(client=client),
    # schema=NanoAODSchema,
    schema=llpnanoaodschema.LLPNanoAODSchema,
    # maxchunks=1,
    skipbadfiles=True,
)

p = sidm_processor.SidmProcessor(
    channels,
    ["BKG_study"], unweighted_hist=True,
)

out_bkg = runner.run(fileset_bkg, treename="Events", processor_instance=p)
out_bkg = out_bkg["out"]

In [ ]:
# adding and saving
out_all = out_sig | out_sig4 | out_bkg 
coffea.util.save(out_all, "outputs/bkg_" + vr + ".coffea")

## Tests:

In [ ]:
print(out_all.keys())

In [ ]:
allsamples = sig_2mu + sig_4mu + bkg
print(allsamples)

sig_leg = [s.split("_")[1] + "  " + s.split("_")[2] + "  " + s.split("_")[3] for s in sig_2mu]
figw, figh = 12, 12

# cols = ["b", "r", "g", "c", "k", "y"]
# labels = ["0.15", "0.17", "0.19", "0.21", "0.23", "0.25"]

In [ ]:
# loading the file
output = coffea.util.load("outputs/bkg_" + vr + ".coffea")
out = output

In [ ]:
npl = 3
plt.subplots(1, npl, figsize=(npl*figw, figh))

plt.subplot(1, npl, 1)
for sss in allsamples:
    print(sss)
    utilities.plot(out[sss]["hists"]["lj_pt"][ch1, ::2j], label = sig_2mu, density=True)
    plt.legend(allsamples, title="Sample", alignment="left", loc=0)
    # plt.title("den")
    plt.ylabel("Arbitrary units")

plt.subplot(1, npl, 2)
for sss in allsamples:
    utilities.plot(out[sss]["hists"]["lj_pt"][ch2, ::2j], label = sig_2mu, density=True)
    plt.legend(allsamples, title="Sample", alignment="left", loc=0)
    plt.ylabel("Arbitrary units")

plt.subplot(1, npl, 3)
for sss in allsamples:
    utilities.plot(out[sss]["hists"]["lj_pt"][ch3, ::2j], label = sig_2mu, density=True)
    plt.legend(allsamples, title="Sample", alignment="left", loc=0)
    # plt.title("den")
    plt.ylabel("Arbitrary units")

In [ ]:
# LJ quantities

# "lj_lj_invmass", "lj_lj_absdphi", "lj_lj_absdR""lj_lj_absdeta"
# , "lj_lj_absdR", "lj_lj_invmass"]
hists_to_plot = ["lj_n","lj0_pt", "lj1_pt"]

for hst in hists_to_plot:
    npl = 3
    plt.subplots(1, npl, figsize=(npl*figw, figh))
    
    plt.subplot(1, npl, 1)
    for sss in allsamples:
        utilities.plot(out[sss]["hists"][hst][ch1, ::2j], density=True)
        plt.legend(allsamples, title="Sample", alignment="left", loc=0)
        # plt.title("den")
    
    plt.subplot(1, npl, 2)
    for sss in allsamples:
        utilities.plot(out[sss]["hists"][hst][ch2, ::2j], density=True)
        plt.legend(allsamples, title="Sample", alignment="left", loc=0)
        # plt.title("den")
    
    plt.subplot(1, npl, 3)
    for sss in allsamples:
        utilities.plot(out[sss]["hists"][hst][ch3, ::2j], density=True)
        plt.legend(allsamples, title="Sample", alignment="left", loc=0)
        # plt.title("den")

In [ ]:
# LJ-LJ quantities

hists_to_plot = ["lj_lj_invmass", "lj_lj_absdR", "lj_lj_absdphi", "lj_lj_absdeta"]

for hst in hists_to_plot:
    npl = 3
    plt.subplots(1, npl, figsize=(npl*figw, figh))
    
    plt.subplot(1, npl, 1)
    for sss in allsamples:
        utilities.plot(out[sss]["hists"][hst][ch1, ::2j], density=True)
        plt.legend(allsamples, title="Sample", alignment="left", loc=0)
        # plt.title("den")
    
    plt.subplot(1, npl, 2)
    for sss in allsamples:
        utilities.plot(out[sss]["hists"][hst][ch2, ::2j], density=True)
        plt.legend(allsamples, title="Sample", alignment="left", loc=0)
        # plt.title("den")
    
    plt.subplot(1, npl, 3)
    for sss in allsamples:
        utilities.plot(out[sss]["hists"][hst][ch3, ::2j], density=True)
        plt.legend(allsamples, title="Sample", alignment="left", loc=0)
        # plt.title("den")